In [ ]:
# @title 🛠️ DRIVE SMART EXTRACTOR (ZIP & GZ) { display-mode: "form" }
import os, re, warnings
from google.colab import drive, auth
from googleapiclient.discovery import build

warnings.filterwarnings("ignore")

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
auth.authenticate_user()

# --- FORM INPUTS ---
# @markdown ### 📂 STORAGE SETTINGS
STORAGE_TYPE = "MyDrive" # @param ["MyDrive", "Shared Drives"]
# @markdown ### 🔗 INPUT FILE (ZIP, GZ, or TAR.GZ)
FILE_URL_OR_ID = "" # @param {type:"string"}
# @markdown ### ⚙️ EXTRACTION SETTINGS
CREATE_OUTPUT_FOLDER = "NO" # @param ["YES", "NO"]
OUTPUT_FOLDER_NAME = "" # @param {type:"string"}

def extract_id(input_value):
    id_pattern = r'(?<=/d/)([a-zA-Z0-9_-]+)'
    match = re.search(id_pattern, input_value)
    if match: return match.group(1)
    return input_value.strip()

def get_real_path(service, folder_id):
    curr_id = folder_id
    path_segments = []
    while True:
        try:
            f = service.files().get(fileId=curr_id, fields='name, parents', supportsAllDrives=True).execute()
            name = f.get('name')
            if name in ['My Drive', 'Drive của tôi']:
                path_segments.insert(0, '/content/drive/MyDrive')
                break
            path_segments.insert(0, name)
            if not f.get('parents'): break
            curr_id = f.get('parents')[0]
        except:
            path_segments.insert(0, '/content/drive/MyDrive')
            break
    return "/".join(path_segments).replace("//", "/")

def start_extraction_process():
    if not FILE_URL_OR_ID:
        print("❌ ERROR: Please enter a File ID or URL.")
        return

    service = build('drive', 'v3')
    file_id = extract_id(FILE_URL_OR_ID)

    try:
        meta = service.files().get(fileId=file_id, fields='name, parents', supportsAllDrives=True).execute()
        fname = meta.get('name').strip()
        parent_id = meta.get('parents')[0]

        print(f"🆔 File identified: {fname}")
        parent_path = get_real_path(service, parent_id)

        # Determine Folder Name
        if CREATE_OUTPUT_FOLDER == "YES":
            # Strip extensions to get folder name
            base_name = fname
            for ext in ['.tar.gz', '.gz', '.zip', '.tgz']:
                if base_name.lower().endswith(ext):
                    base_name = base_name[: -len(ext)]
                    break

            final_folder = OUTPUT_FOLDER_NAME if OUTPUT_FOLDER_NAME.strip() else base_name
            target_path = os.path.join(parent_path, final_folder)
            if not os.path.exists(target_path):
                os.makedirs(target_path)
        else:
            target_path = parent_path

        full_input_path = os.path.join(parent_path, fname)

        if os.path.exists(full_input_path):
            print(f"⚡ Processing extraction to: {target_path}")

            # --- SELECTION LOGIC BASED ON EXTENSION ---
            if fname.lower().endswith('.zip'):
                !unzip -o -q "{full_input_path}" -d "{target_path}"

            elif fname.lower().endswith('.tar.gz') or fname.lower().endswith('.tgz'):
                !tar -xzf "{full_input_path}" -C "{target_path}"

            elif fname.lower().endswith('.gz'):
                # For single .gz files, we use gunzip.
                # Note: gunzip usually extracts in-place, so we copy it to target first if needed
                if CREATE_OUTPUT_FOLDER == "YES":
                    dest_file = os.path.join(target_path, fname)
                    !cp "{full_input_path}" "{dest_file}"
                    !gunzip -f "{dest_file}"
                else:
                    !gunzip -k -f "{full_input_path}" # -k keeps the original

            else:
                print(f"❓ Unknown extension. Attempting general extraction...")
                !7z x "{full_input_path}" -o"{target_path}" -y

            print("-" * 40)
            print(f"🎉 SUCCESS! Location: {target_path}")
        else:
            print(f"❌ ERROR: File not found at {full_input_path}")

    except Exception as e:
        print(f"❌ SYSTEM ERROR: {e}")

start_extraction_process()